In [1]:
# ==============================================================================
# HOSPITAL SANDBOX RUN: VARIATIONAL QUANTUM NEURAL NETWORK TRAINING PIPELINE
# ==============================================================================

# 1. Mount Google Drive First
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install Required Quantum Framework Dependencies
!pip install pennylane --quiet

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import pennylane as qml

# Set random seeds for strict reproducible training runs
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ==============================================================================
# STEP 3: ESTABLISH SANDBOX VERIFIED DIRECTORY PATHS
# ==============================================================================
SANDBOX_DIR = "/content/drive/MyDrive/(1)Hybrid_QML_CVD_Project_HospitalRun"
FEATURES_DIR = os.path.join(SANDBOX_DIR, "features")
MODELS_DIR = os.path.join(SANDBOX_DIR, "models")

# Ensure structural folder boundaries exist
os.makedirs(MODELS_DIR, exist_ok=True)

# ==============================================================================
# STEP 4: FEATURE INGESTION & SUB-SAMPLING (CRITICAL FOR PENNYLANE SPEED)
# ==============================================================================
print("🔄 Ingesting extracted 1D-CNN features from sandbox storage...")

X_features = np.load(os.path.join(FEATURES_DIR, "cnn_features.npy"))
y_labels = np.load(os.path.join(FEATURES_DIR, "cnn_labels.npy"))

print(f"📊 Full Ingested Latent Feature Shape: {X_features.shape}")
print(f"📊 Full Ingested Targets Vector Shape: {y_labels.shape}")

# PENNYLANE ACCELERATION TRIMMING: State-vector simulators track complex wave mechanics.
# We sub-sample a stable subset while maintaining the exact distribution of class states.
SUBSET_SIZE = 3000

if len(X_features) > SUBSET_SIZE:
    print(f"\n✂️ Sub-sampling {SUBSET_SIZE} points via stratified distribution for fast execution...")
    _, X_qnn, _, y_qnn = train_test_split(
        X_features, y_labels, test_size=SUBSET_SIZE/len(X_features), random_state=42, stratify=y_labels
    )
else:
    X_qnn, y_qnn = X_features, y_labels

# ==============================================================================
# STEP 5: PYTORCH DATALOADER CONFIGURATION
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_qnn, y_qnn, test_size=0.2, random_state=42, stratify=y_qnn
)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==============================================================================
# STEP 6: DEFINE PENNYLANE QUANTUM CIRCUIT ARCHITECTURE
# ==============================================================================
num_qubits = 4  # Dimensions mapped dynamically to qubits
num_layers = 2  # Variational Ansatz layer configuration depths

dev = qml.device("default.qubit", wires=num_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    # Angle Embedding: Encodes classical feature dimensions into quantum states
    qml.AngleEmbedding(inputs, wires=range(num_qubits), rotation='X')

    # Variational Layers (Ansatz Topology configuration)
    for l in range(num_layers):
        for i in range(num_qubits):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RY(weights[l, i, 1], wires=i)
            qml.RZ(weights[l, i, 2], wires=i)

        # Entangling Ring Layout
        for i in range(num_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        if num_qubits > 2:
            qml.CNOT(wires=[num_qubits - 1, 0])

    # Measure expectation values across PauliZ properties back to classical tensors
    return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

# ==============================================================================
# STEP 7: BUILD HYBRID QUANTUM NEURAL NETWORK CLASS (QNN)
# ==============================================================================
class HybridQNNClassifier(nn.Module):
    def __init__(self):
        super(HybridQNNClassifier, self).__init__()
        # Pre-quantum Layer: Maps 16-dimensional CNN traits to 4 qubits
        self.pre_quantum_dense = nn.Linear(16, num_qubits)

        # Pennylane Quantum Variational Torch Layer Hook
        weight_shapes = {"weights": (num_layers, num_qubits, 3)}
        self.quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

        # Post-quantum Classifier Layer: Maps 4 qubit properties to 5 target classes
        self.post_quantum_dense = nn.Linear(num_qubits, 5)

    def forward(self, x):
        x = torch.tanh(self.pre_quantum_dense(x))
        x = self.quantum_layer(x)
        logits = self.post_quantum_dense(x)
        return logits

# Initialize Model Deployment Variables
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qnn_model = HybridQNNClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(qnn_model.parameters(), lr=0.01)

# ==============================================================================
# STEP 8: QUANTUM MODEL TRAINING EXECUTION LOOP
# ==============================================================================
epochs = 5
print(f"\n🔮 Initiating Quantum VQC Training Loop on target device: {device}...")

for epoch in range(epochs):
    qnn_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = qnn_model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100
    print(f" ⚛️ Quantum Epoch [{epoch+1:02d}/{epochs:02d}] ➔ Loss: {epoch_loss:.4f} | QNN Accuracy: {epoch_acc:.2f}%")

# Save finalized Quantum Checkpoint Weights to Sandbox
torch.save(qnn_model.state_dict(), os.path.join(MODELS_DIR, "best_quantum_model.pt"))
print("\n💾 Variational Quantum Weights Saved Successfully to Sandbox Folder!")

print("\n" + "="*65)
print(" 🎉 SUCCESS: QUANTUM PIPELINE EXECUTION FOR HOSPITAL RUN FINISHED")
print("="*65)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 792.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 56.2 MB/s eta 0:00:00
🔄 Ingesting extracted 1D-CNN features from sandbox storage...
📊 Full Ingested Latent Feature Shape: (112163, 16)
📊 Full Ingested Targets Vector Shape: (112163,)

✂️ Sub-sampling 3000 points via stratified distribution for fast execution...

🔮 Initiating Quantum VQC Training Loop on target device: cpu...
 ⚛️ Quantum Epoch [01/05] ➔ Loss: 0.7579 | QNN Accuracy: 72.75%
 ⚛️ Quantum Epoch [02/05] ➔ Loss: 0.4437 | QNN Accura